# 01 - Portfolio exploration

Synthetic digital-lending portfolio, snapshot 2025-12-31. This notebook reads the SQL marts built by `make etl`; definitions are in `docs/metrics.md`.

In [1]:
import duckdb
import pandas as pd

from credit_risk.config import get_settings

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.4f}".format)
settings = get_settings()
con = duckdb.connect(str(settings.paths.warehouse), read_only=True)
q = lambda sql: con.execute(sql).df()  # noqa: E731

## Portfolio shape
One row per loan in `marts.loan_repayment_summary`.

In [2]:
loans = q("SELECT * FROM marts.loan_repayment_summary")
loans.groupby("product_type").agg(
    loans=("loan_id", "size"), avg_amount=("loan_amount", "mean"), avg_term=("term_days", "mean")
)

,loans,avg_amount,avg_term
product_type,,,
emergency_loan,3072,"2,136.5560",7.0000
micro_loan,5687,"7,394.1445",21.8973
salary_advance,4014,"21,244.4444",30.0000
sme_working_capital,2347,"29,902.8547",75.0959


## Headline KPIs recomputed in Python
The same functions the dashboard uses; `tests/test_marts.py` asserts they match the SQL marts.

In [3]:
from credit_risk.analytics import metrics as lm

pd.Series(lm.portfolio_kpis(loans, settings.risk, pd.Timestamp(settings.generator.snapshot_date)))

total_borrowers                 4,975.0000
active_borrowers                  978.0000
total_loans                    15,120.0000
active_loans                      991.0000
total_disbursed           204,071,200.0000
outstanding_principal      13,844,019.9300
total_collections         210,959,694.0800
repayment_rate                      0.9646
on_time_collection_rate             0.7785
completion_rate                     0.9135
default_rate                        0.0829
bad_30dpd_rate                      0.1284
par1                                0.2846
par7                                0.2512
par30                               0.1997
par60                               0.1469
par90                               0.1084
delinquency_rate                    0.5399
avg_dpd                            75.3869
median_dpd                         67.0000
max_dpd                           179.0000
dtype: float64

## PAR trend (SQL)

In [4]:
q((settings.paths.sql / "analytics/03_par_by_month.sql").read_text()).tail(12)

,month_end,outstanding_principal,active_loans,par1,par7,par30,par60,par90
12,2025-01-31,"10,368,556.0000",782,0.2010,0.1854,0.1343,0.1055,0.0661
13,2025-02-28,"10,498,575.0000",815,0.2063,0.1763,0.1224,0.0972,0.0742
14,2025-03-31,"11,247,323.0000",813,0.1657,0.1464,0.1130,0.0798,0.0621
15,2025-04-30,"12,464,384.0000",886,0.1785,0.1515,0.1006,0.0770,0.0632
16,2025-05-31,"12,101,263.0000",903,0.1574,0.1451,0.0894,0.0616,0.0437
17,2025-06-30,"12,461,234.0000",938,0.1646,0.1430,0.0922,0.0623,0.0402
18,2025-07-31,"13,597,876.0000",988,0.1879,0.1683,0.1027,0.0642,0.0435
19,2025-08-31,"15,067,116.0000",1078,0.2090,0.1824,0.1240,0.0751,0.0472
20,2025-09-30,"15,184,138.0000",1112,0.2499,0.2140,0.1353,0.0936,0.0534
21,2025-10-31,"14,805,185.0000",1142,0.2369,0.1983,0.1455,0.1104,0.0794


## DPD distribution of the on-book portfolio

In [5]:
q((settings.paths.sql / "analytics/04_dpd_distribution.sql").read_text())

,dpd_bucket,loans,outstanding_principal,principal_share,avg_dpd
0,current,456,"9,903,487.0000",0.7154,0.0000
1,1-6,42,"462,619.0000",0.0334,3.3000
2,7-29,101,"713,172.0000",0.0515,17.6000
3,30-59,97,"730,969.0000",0.0528,43.1000
4,60-89,82,"533,248.0000",0.0385,73.1000
5,90+,213,"1,500,524.0000",0.1084,132.6000


**Reading.** Performing short-tenor loans leave the book within weeks, so the on-book stock over-represents delinquent loans. Flow measures - the 30+ DPD rate of each vintage - are the better gauge of new-business quality:

In [6]:
q((settings.paths.sql / "analytics/12_cohort_vintage_analysis.sql").read_text())

,disbursement_month,loans,mob_1,mob_2,mob_3,mob_6,default_mob_6
0,2024-01-01,392,0.2931,0.8966,0.9655,1.0000,1.0000
1,2024-02-01,506,0.3117,0.8701,0.9221,1.0000,1.0000
2,2024-03-01,738,0.3333,0.8750,0.9479,1.0000,1.0000
3,2024-04-01,828,0.4407,0.8729,0.9576,1.0000,1.0000
4,2024-05-01,652,0.4146,0.8780,0.9146,1.0000,1.0000
5,2024-06-01,559,0.4306,0.8333,0.9167,1.0000,1.0000
6,2024-07-01,579,0.3421,0.8026,0.8947,1.0000,1.0000
7,2024-08-01,535,0.3500,0.8167,0.8333,1.0000,1.0000
8,2024-09-01,533,0.4194,0.9194,0.9677,1.0000,1.0000
9,2024-10-01,551,0.2464,0.7681,0.8551,1.0000,1.0000


## Roll rates
Row = bucket at month-end M, column = state at M+1.

In [7]:
q((settings.paths.sql / "analytics/11_roll_rate_matrix.sql").read_text())

,from_state,current,1-6,7-29,30-59,60-89,90+,repaid,written_off
0,current,0.2357,0.0419,0.1231,0.0070,NaN,NaN,0.5923,NaN
1,1-6,NaN,NaN,0.0011,0.3959,NaN,NaN,0.6029,NaN
2,7-29,NaN,NaN,NaN,0.5768,0.0179,NaN,0.4053,NaN
3,30-59,NaN,NaN,NaN,0.0007,0.7833,0.0241,0.1920,NaN
4,60-89,NaN,NaN,NaN,NaN,0.0027,0.8750,0.1223,NaN
5,90+,NaN,NaN,NaN,NaN,NaN,0.6832,0.0125,0.3043
